---

# Determina a quantidade de relâmpagos por cidade e estado para um determinado dia.

---

- `OBJETIVO`:
> Este código baixa os dados de 20s flashes do sensor GLM do satélite GOES-16 ou GOES-19 do repositório da AMAZON, extrai os relâmpagos para o Brasil, gera arquivo netcdf e plota o mapa diário, extrai a quantidade de relâmpagos para cada município e estado brasileiro.  

- `DADOS`:
> Dados do sensor GLM do satélite [GOES-16](https://noaa-goes16.s3.amazonaws.com/index.html#GLM-L2-LCFA/) ou [GOES-19](https://noaa-goes19.s3.amazonaws.com/index.html#GLM-L2-LCFA/) fornecido pela AMAZON. Exemplo de nome do arquivo: `OR_GLM-L2-LCFA_G16_s20201820000000_e20201820000200_c20201820000224.nc`

- `OBSERVAÇÕES`:
    > Em **1 dia** de dados temos `4320 arquivos` e demora `1h38s` para baixar.

- `REALIZADO POR`:
> Enrique V. Mattos - 06/10/2026

- `ATUALIZADO POR`:
> Enrique V. Mattos - 06/10/2026
---



# **1° Passo:** Preparando ambiente

In [1]:
# instala e importa bibliotecas
!pip install -q boto3 ultraplot cartopy geobr salem rasterio pyproj geopandas

# importa bibliotecas
import geopandas as gpd
import os
import time
from datetime import timedelta, datetime
import os
import boto3
import geobr
from botocore import UNSIGNED
from botocore.config import Config
import pandas as pd
import glob
import ultraplot as uplt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
import salem
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# monta drive
from google.colab import drive
drive.mount('/content/drive')

# limites do Brasil
lonmin_BR, lonmax_BR, latmin_BR, latmax_BR = -75.0, -33.0, -35.0, 7.0

# diretório raiz
dir = '/content/drive/MyDrive/PYHTON/00_GITHUB/000_CODIGOS_REFERENCIA/03_RELAMPAGOS_SATELITE_REFERENCIA'

# diretório de saída
dir_output = f'{dir}/output/GLM_08'
dir_glm20s = f'{dir_output}/01_glm_20s_goes'
dir_tabela_diaria = f'{dir_output}/02_glm_diario_goes'
dir_estatisticas_estado_e_municipio = f'{dir_output}/03_estatisticas_estado_e_municipio'

# cria pasta de saída
os.makedirs(dir_output, exist_ok=True)
os.makedirs(dir_tabela_diaria, exist_ok=True)
os.makedirs(dir_estatisticas_estado_e_municipio, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.6 MB/s eta 0:00:00
Mounted at /content/drive


# **2° Passo:** Declarando funções

In [2]:
#==================================================================================================#
#                   FUNÇÃO QUE FAZ O DOWNLOAD DOS DADOS DE 20S DA AWS
#==================================================================================================#
def download_GLM20s_AWS(yyyymmddhhmnss, goes_number, path_dest):

    ''' Função para baixar os arquivos netcdf de 20s do GLM processado e disponibilizado pela Amazon Web Service (AWS)

    Parâmetros:
               yyyymmddhhmnss (string): ano+mes+dia+hora+minuto+segundo. Exemplo: 20251113033020
               goes_number (string): número do satélite GOES: 16 ou 19
               path_dest (string): nome do direório + nome do arquivo que foi baixado

    Retorna:
            file_name (string): nome do direório + nome do arquivo que foi baixado

    Observação: acesso aos dados
                    GOES-16: https://noaa-goes16.s3.amazonaws.com/index.html#ABI-L2-CMIPF/
                    GOES-19: https://noaa-goes19.s3.amazonaws.com/index.html#ABI-L2-CMIPF/
    '''

    year = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%Y')
    day_of_year = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%j')
    hour = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%H')
    min = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%M')
    seg = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%S')

    # informação do repositório da AMAZON. Exemplo: 'noaa-goes19'
    bucket_name = f'noaa-goes{goes_number}'

    # inicializa o S3 cliente
    s3_client = boto3.client('s3', config=Config(signature_version=UNSIGNED))

    # estutura do arquivo. Exemplo: "GLM-L2-LCFA/2025/317/03/OR_GLM-L2-LCFA_G19_s20253170330200"
    product_name = "GLM-L2-LCFA"
    prefix = f'{product_name}/{year}/{day_of_year}/{hour}/OR_{product_name}_G{goes_number}_s{year}{day_of_year}{hour}{min}{seg}'

    # pesquisa pelo arquivo no servidor
    s3_result = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix, Delimiter = "/")

    # verifica se o arquivo esta diponível
    #-----------------------------------------------------------------------------------------------------------
    if 'Contents' not in s3_result:

        # se o arquivo não esta disponível
        print(f'No files found for the date: {yyyymmddhhmnss}, Product-{product_name}')
        return -1
    else:

        # se o arquivo existe
        for obj in s3_result['Contents']:
            key = obj['Key']

            # imprime o nome do arquivo
            file_name = key.split('/')[-1].split('.')[0]

        # baixa o arquivo
        if os.path.exists(f'{path_dest}/{file_name}.nc'):
            print(f'File {path_dest}/{file_name}.nc exists')
        else:
            print(f'Downloading file {path_dest}/{file_name}.nc')
            s3_client.download_file(bucket_name, key, f'{path_dest}/{file_name}.nc')

    return f'{file_name}'

# Função que calcula o índice i e j da localização do relâmpago
def index(longitudes_matriz, latitudes_matriz, lon_raio, lat_raio):

    ''' Função para calcular o índice (i e j) do pixel de uma matriz que o relâmpago pertence

    Parâmetros:
               longitudes_matriz (array): array de uma dimensão das longitudes da matriz em graus
               latitudes_matriz (array): array de uma dimensão das latitudes da matriz em graus
               lon_raio (float): valor da longitude do relâmpago em graus
               lat_raio (float): valor da latitude do relâmpago em graus

    Retorna:
            indice_lat_raio (float): índice da latitude (ou seja, da linha) do pixel da matriz que o relâmpago pertence
            indice_lon_raio (float): índice da longitude (ou seja, da coluna) do pixel da matriz que o relâmpago pertence
    '''

    # calcula a diferença entre as lats/lons da matriz e a latitude/longitude do relâmpago
    distancia_lon = (longitudes_matriz - lon_raio)**2
    distancia_lat = (latitudes_matriz - lat_raio)**2

    # índice da longitude e latitude do relâmpago
    indice_lon_raio = np.nonzero(distancia_lon == np.min(distancia_lon))
    indice_lat_raio = np.nonzero(distancia_lat == np.min(distancia_lat))

    # retorna os valores dos índices calculados
    return indice_lat_raio, indice_lon_raio

# **PARTE 1:** DEFINE O DIA

In [3]:
#========================================================================================================================#
#                                          DEFINE A DATA DO ARQUIVO
#========================================================================================================================#
# data INICIAL
anoi, mesi, diai, hori, mini, segi = '2026', '10', '04', '00', '00', '00'

# data FINAL
anof, mesf, diaf, horf, minf, segf = anoi, mesi, diai, '23', '59', '59'

# **PARTE 2:** DOWNLOAD DOS DADOS - Wall time: 30min 3s

In [ ]:
%%time
#========================================================================================================================#
#                                          CRIAS AS PASTAS POR HORÁRIO
#========================================================================================================================#
# cria as pastas de saida: ano-mes-dia/hora
for data in pd.date_range(f'{anoi}-{mesi}-{diai} {hori}:{mini}', f'{anof}-{mesf}-{diaf} {horf}:{minf}', freq='1H'):

    # extrai ano e mês
    anox = data.strftime('%Y')
    mesx = data.strftime('%m')
    diax = data.strftime('%d')
    horx = data.strftime('%H')

    # cria pasta daquele horário
    os.makedirs(f'{dir_glm20s}/{anox}-{mesx}-{diax}/{horx}', exist_ok=True)

#========================================================================================================================#
#                                                DOWNLOAD DOS DADOS
#========================================================================================================================#
# data INICIAL junta como uma string
date_ini = str(datetime(int(anoi),int(mesi),int(diai),int(hori),int(mini)))

# data FINAL junta como uma string
date_end = str(datetime(int(anof),int(mesf),int(diaf),int(horf),int(minf)))

# data inicial é fixada como a data do loop
date_loop = date_ini

# loop nos arquivos do GLM
while (date_loop <= date_end):

    # data
    yyyymmddhhmnss = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%Y%m%d%H%M%S')

    # ano, mes, dia, hora, minuto e segundos
    ano = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%Y')
    mes = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%m')
    dia = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%d')
    hor = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%H')
    min = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%M')
    seg = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%S')

    # define se GOES-16 ou GOES-19
    start_g19 = datetime(2025,4,7,0,0)
    imagem_atual = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S')
    goes_number = '16' if imagem_atual < start_g19 else '19'
    print(f'DOWNLOAD DO ARQUIVO GLM ===>>> {ano}-{mes}-{dia} {hor}:{min}:{seg}')

    # local onser será baixado o arquivo
    local_salvar_arquivo = f'{dir_glm20s}/{ano}-{mes}-{dia}/{hor}'

    # download o arquivo
    file_glm20s = download_GLM20s_AWS(yyyymmddhhmnss, goes_number, local_salvar_arquivo)

    # incrementa a variável the date_loop
    date_loop = str(datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S') + timedelta(seconds=20))
    print('\n')

# **PARTE 2:** SELECIONA OS RELÂMPAGOS QUE OCORRERAM NO BRASIL

In [ ]:
%%time
# lista os arquivos
files = sorted(glob.glob(f'{dir_glm20s}/{anoi}-{mesi}-{diai}/*/*.nc'))

# Lista para armazenar dados filtrados (mais eficiente que DataFrames)
filtered_data = []

# Otimização: pré-compilar condições de filtro
def in_region(lon, lat):
    return (lon > lonmin_BR) & (lon < lonmax_BR) & (lat > latmin_BR) & (lat < latmax_BR)

# Loop
for file in files:

    print(f'Processando: {file}')

    # Usando context manager para fechar arquivo automaticamente
    with xr.open_dataset(file) as glm_20s:

        # Acessando arrays diretamente (mais rápido que via DataFrame)
        lons_flash = glm_20s['flash_lon'].values
        lats_flash = glm_20s['flash_lat'].values
        area_flash = glm_20s['flash_area'].values # m2
        energia_flash = glm_20s['flash_energy'].values # Joule
        time_first_event_flash = glm_20s['flash_time_offset_of_first_event'].values # segundos
        time_last_event_flash = glm_20s['flash_time_offset_of_last_event'].values # segundos
        quality_flash = glm_20s['flash_quality_flag'].values # [0,1,2,3]

        # Máscara booleana vetorizada (mais rápido que filtro no DataFrame)
        mask = in_region(lons_flash, lats_flash)

        # Apenas processa se houver dados na região
        if np.any(mask):

            # Cria DataFrame apenas com dados filtrados
            df_filtered = pd.DataFrame({ 'time_first_event(s)': time_first_event_flash[mask],
                                         'time_last_event(s)': time_last_event_flash[mask],
                                         'lat': lats_flash[mask],
                                         'lon': lons_flash[mask],
                                         'area(m2)': area_flash[mask],
                                         'energia(Joule)': energia_flash[mask],
                                         'qualidade': quality_flash[mask]})

            filtered_data.append(df_filtered)

# concatenação final
df_flash_total = pd.concat(filtered_data, ignore_index=True) if filtered_data else pd.DataFrame(columns=['time_first_event(s)',
                                                                                                         'time_last_event(s)',
                                                                                                         'lat',
                                                                                                         'lon',
                                                                                                         'area(m2)',
                                                                                                         'energia(Joule)',
                                                                                                         'qualidade'])

# calcula a duração em segundos. Transforma a coluna 'time_first_event(s)' e 'time_last_event(s)' para datetime e depois calcula a diferença em segundos
df_flash_total['duracao(s)'] = pd.to_datetime(df_flash_total['time_last_event(s)'], unit='s') - pd.to_datetime(df_flash_total['time_first_event(s)'], unit='s')

# transformar para float
df_flash_total['duracao(s)'] = df_flash_total['duracao(s)'].dt.total_seconds()

# salvar dataframe
df_flash_total.to_csv(f'{dir_tabela_diaria}/flash_glm_goes_{anoi}-{mesi}-{diai}.csv', index=False)

# coloca a coluna da qualidade na última posição
col_qualidade = df_flash_total.pop('qualidade')
df_flash_total.insert(len(df_flash_total.columns), 'qualidade', col_qualidade)

# mostra o dataframe final com a coluna time, lat e lon
df_flash_total

# **PARTE 3:** GERA ARQUIVO NETCDF DIÁRIO


In [ ]:
%%time
# espaçamento da grade
delta = 0.08   # grade com 8 km de resolução espacial

# montando a grade
lons = np.arange(lonmin_BR, lonmax_BR, delta)
lats = np.arange(latmin_BR, latmax_BR, delta)

# quantidade de pontos para longitude e latitude
nlon = len(lons)
nlat = len(lats)

# transforma de dataframe para numpy.array
flash_lon, flash_lat = df_flash_total['lon'].values, df_flash_total['lat'].values

# declara a matriz de relâmpagos
flashes = np.zeros((nlat, nlon))  # Exemplo: flash = np.zeros(qte_linha, qte_coluna)

# loop em cada longitude e latitude da lista
for lonraio, latraio in zip(flash_lon, flash_lat):

    # função que extrai a qual pixel (ou seja, determina as variáveis 'lin' e 'col') aquele relâmpago pertence
    lin, col = index(lons, lats, lonraio, latraio)

    # soma os relâmpagos por pixel
    flashes[lin,col]+=1

# gera netcdf
data_vars = {'flash':(('lat', 'lon'), flashes, {'units': 'ocorrências/64km²', 'long_name':'Flashes'})}
coords = {'lat': lats, 'lon': lons}
ds = xr.Dataset(data_vars=data_vars, coords=coords)
ds.to_netcdf(f'{dir_tabela_diaria}/flash_glm_goes_{anoi}-{mesi}-{diai}.nc')

# **PARTE 4:** PLOTA MAPA DO ARQUIVO NETCDF DIÁRIO

In [ ]:
%%time
#==================================================================================================#
#                                 LEITURA DO ARQUIVO NETCDF DIÁRIO
#==================================================================================================#
glm_amazon_diario = xr.open_dataset(f'{dir_tabela_diaria}/flash_glm_goes_{anoi}-{mesi}-{diai}.nc').sel(lon=slice(lonmin_BR, lonmax_BR),
                                                                                         lat=slice(latmin_BR, latmax_BR))

#==================================================================================================#
#                                 DEFINIÇÕES DO GRÁFICO
#==================================================================================================#
# extensão da imagem [min. lon, min. lat, max. lon, max. lat]
extent = [lonmin_BR, latmin_BR, lonmax_BR, latmax_BR]

# cria moldura da figura
fig, ax = uplt.subplots(axheight=6.9, axwidth=6.8, tight=True, proj='pcarree')

# formata os eixos
ax.format(coast=True, borders=True, innerborders=False,
          labels=True, latlines=20, lonlines=20,
          latlim=(latmin_BR, latmax_BR), lonlim=(lonmin_BR, lonmax_BR),
          small='20px', large='25px',
          title=f'Acumulado de Relâmpagos \n',
          titleloc='l',
          titleweight='bold',
          titlecolor='bright red')

# plota subtítulo
ax.text(0.000, 1.035,
        f'Satélite: GOES-19 (8km) | Sensor: GLM | Período: {anoi}-{mesi}-{diai}',
        transform = ax.transAxes,
        color = 'gray',
        fontsize = 9,
        verticalalignment = 'top')

#==================================================================================================#
#                                       PLOTA FIGURA
#==================================================================================================#
# quantidade de flashes por dia para a área TOTAL da imagem plotada
total = int(glm_amazon_diario['flash'].sum(('lat', 'lon')))

# substituir valores "zero" por "NaN"
glm_amazon_diario = glm_amazon_diario.where(glm_amazon_diario != 0, np.nan)

# figura
# aqui dividimos a matriz por 64 para termos a unidade em km2. Lembre-se que
# a matriz de flashes tem tamanho de 8 km x 8 km (ou seja, área de 64 km2).
map1 = ax.contourf(glm_amazon_diario['lon'],
                   glm_amazon_diario['lat'],
                   glm_amazon_diario['flash']/64.,
                   cmap='jet',
                   levels=uplt.arange(0, 5, 1),
                   extend='max')

# total de relâmpagos
ax.text(lonmax_BR-14., latmin_BR+0.5, f'Total: {int(total)} relâmpagos', color='red', fontsize=13)

# plota contornos dos Estados
shapefile = list(shpreader.Reader('https://github.com/evmpython/Minicurso_UFMS_SEMADESC_marco_2026/raw/main/01_utils/BR_UF_2019.shp').geometries())
ax.add_geometries(shapefile, ccrs.PlateCarree(), edgecolor='grey', facecolor='none', linewidth=0.5, zorder=2)

# barra de cores
fig.colorbar(map1, loc='b', label='relâmpagos/$km^{2}$', ticks=1, ticklabelsize=15, labelsize=15, space=1.4, width=0.3)

# salva figura
fig.save(f'{dir_tabela_diaria}/mapa_diario_glm_flash_goes19_{anoi}-{mesi}-{diai}.jpg', dpi=300)

# **PARTE 5:** SELECIONA RELÂMPAGOS POR ESTADO

In [4]:
%%time
#========================================================================================================================#
#                                                  CARREGA SHAPEFILES
#========================================================================================================================#
print("Carregando shapefile dos estados brasileiros...")
shapefile_estados = geobr.read_state(year=2020)

#========================================================================================================================#
#                                          LEITURA E PREPARAÇÃO DOS FLASHES
#========================================================================================================================#
print(f"Lendo arquivo de flashes do dia {anoi}-{mesi}-{diai}...")
df_flash = pd.read_csv(f'{dir_tabela_diaria}/flash_glm_goes_{anoi}-{mesi}-{diai}.csv')

# Filtra os limites do Brasil
df_flash_filtered = df_flash[(df_flash['lon'] > lonmin_BR) & (df_flash['lon'] < lonmax_BR) &
                             (df_flash['lat'] > latmin_BR) & (df_flash['lat'] < latmax_BR)]

# Converte os flashes filtrados para GeoDataFrame
gdf_flashes = gpd.GeoDataFrame(df_flash_filtered,
                               geometry=gpd.points_from_xy(df_flash_filtered.lon, df_flash_filtered.lat), crs='EPSG:4326')

# Garante que ambos os conjuntos de dados estejam na mesma projeção
shapefile_estados = shapefile_estados.to_crs(gdf_flashes.crs)

#========================================================================================================================#
#                     JUNÇÃO ESPACIAL EFICIENTE (Spatial Join) E CONTABILIZAÇÃO POR ESTADO
#========================================================================================================================#
print("Realizando cruzamento espacial dos flashes com os estados...")
# 'sjoin' associa cada ponto de flash ao estado correspondente espacialmente
flashes_com_estado = gpd.sjoin(gdf_flashes, shapefile_estados, how='inner', predicate='within')

# Agrupa os resultados por estado para contar os flashes
print("Agrupando e contabilizando os flashes...")
contagem_flashes_estado = flashes_com_estado.groupby(['abbrev_state']).size().reset_index(name='Total de Flashes')
contagem_flashes_estado.rename(columns={'abbrev_state': 'Estado'}, inplace=True)

# Cria uma base de todos os estados para garantir que estados com zero relâmpagos também apareçam
todos_estados = pd.DataFrame({'Estado': shapefile_estados['abbrev_state']})

# Mescla todos os estados com a contagem para preencher com zero onde não houve registro de flashes
df_flash_counts_by_state = pd.merge(todos_estados, contagem_flashes_estado, on='Estado', how='left')
df_flash_counts_by_state['Total de Flashes'] = df_flash_counts_by_state['Total de Flashes'].fillna(0).astype(int)

# Organizar do maior para o menor da coluna "Total de Flashes"
df_flash_counts_by_state = df_flash_counts_by_state.sort_values(by='Total de Flashes', ascending=False).reset_index(drop=True)

#========================================================================================================================#
#                                       SALVA ARQUIVO CSV FINAL
#========================================================================================================================#
caminho_salvar_estados = f'{dir_estatisticas_estado_e_municipio}/flash_counts_by_state_{anoi}-{mesi}-{diai}.csv'
df_flash_counts_by_state.to_csv(caminho_salvar_estados, index=False)
print(f"Planilha de estados salva em: {caminho_salvar_estados}")

# Mostra os dados
df_flash_counts_by_state

Carregando shapefile dos estados brasileiros...


states_2020_simplified.parquet: 100%|██████████| 1.72M/1.72M [00:00<00:00, 29.5MB/s]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lendo arquivo de flashes do dia 2026-10-04...
Realizando cruzamento espacial dos flashes com os estados...
Agrupando e contabilizando os flashes...
Planilha de estados salva em: /content/drive/MyDrive/PYHTON/00_GITHUB/000_CODIGOS_REFERENCIA/03_RELAMPAGOS_SATELITE_REFERENCIA/output/GLM_08/03_estatisticas_estado_e_municipio/flash_counts_by_state_2026-10-04.csv
CPU times: user 14 s, sys: 1.48 s, total: 15.5 s
Wall time: 26.7 s


,Estado,Total de Flashes
0,MS,492040
1,MT,249881
2,AM,137358
3,PR,80255
4,PA,70121
5,RS,45111
6,GO,44416
7,MG,35207
8,SP,31897
9,TO,31315


In [5]:
# exemplo da tabela dos flashes
df_flash

,time_first_event(s),time_last_event(s),lat,lon,area(m2),energia(Joule),qualidade,duracao(s)
0,2026-10-03 23:59:59.498741627,2026-10-03 23:59:59.568552018,-23.985142,-56.362164,2.212727e+08,1.428509e-14,0.0,0.069810
1,2026-10-03 23:59:59.301137448,2026-10-03 23:59:59.573511124,-27.546303,-53.699097,1.451244e+08,4.285134e-15,0.0,0.272374
2,2026-10-03 23:59:59.067292691,2026-10-03 23:59:59.580377579,-31.954168,-51.316578,3.598352e+08,1.012847e-13,0.0,0.513085
3,2026-10-03 23:59:59.399176598,2026-10-03 23:59:59.645609856,-26.023882,-55.228820,2.279872e+08,2.728504e-14,0.0,0.246433
4,2026-10-03 23:59:59.530404091,2026-10-03 23:59:59.665446759,-20.089750,-50.592533,1.498550e+08,3.285138e-15,0.0,0.135043
...,...,...,...,...,...,...,...,...
1851925,2026-10-04 23:59:18.958957672,2026-10-04 23:59:19.185934066,-21.385414,-55.999123,1.545857e+08,7.285122e-15,0.0,0.226976
1851926,2026-10-04 23:59:18.999393463,2026-10-04 23:59:19.186315536,-19.802502,-46.556076,9.438425e+08,4.862832e-13,0.0,0.186922
1851927,2026-10-04 23:59:18.611814498,2026-10-04 23:59:19.245826721,-19.896223,-58.925790,1.119487e+09,2.752840e-13,0.0,0.634012
1851928,2026-10-04 23:59:19.047458648,2026-10-04 23:59:19.262229919,-20.111134,-61.907600,1.468030e+08,3.028503e-14,0.0,0.214771


# **PARTE 6:** SELECIONA RELÂMPAGOS POR MUNICÍPIO

In [6]:
%%time
#========================================================================================================================#
#                               CARREGA MUNICÍPIOS DE TODO O BRASIL
#========================================================================================================================#
print("Carregando shapefile de todos os municípios do Brasil...")
shapefile_municipios = geobr.read_municipality(year=2020, code_muni=None)

#========================================================================================================================#
#                                   LEITURA E PREPARAÇÃO DOS FLASHES
#========================================================================================================================#
# Carrega a tabela de flashes gerada no passo anterior (usando as variáveis globais anoi, mesi, diai)
print(f"Lendo arquivo de flashes do dia {anoi}-{mesi}-{diai}...")
df_flash = pd.read_csv(f'{dir_tabela_diaria}/flash_glm_goes_{anoi}-{mesi}-{diai}.csv')

# Filtra os limites do Brasil
df_flash_filtered = df_flash[(df_flash['lon'] > lonmin_BR) & (df_flash['lon'] < lonmax_BR) &
                             (df_flash['lat'] > latmin_BR) & (df_flash['lat'] < latmax_BR)]

# Converte os flashes filtrados para GeoDataFrame
gdf_flashes = gpd.GeoDataFrame(df_flash_filtered,
                               geometry=gpd.points_from_xy(df_flash_filtered.lon, df_flash_filtered.lat), crs='EPSG:4326')

# Garante que ambos os conjuntos de dados estejam na mesma projeção
shapefile_municipios = shapefile_municipios.to_crs(gdf_flashes.crs)

#========================================================================================================================#
#                     JUNÇÃO ESPACIAL EFICIENTE (Spatial Join) E CONTABILIZAÇÃO
#========================================================================================================================#
print("Realizando cruzamento espacial dos flashes com os municípios...")
# 'sjoin' associa cada ponto de flash ao município correspondente espacialmente
flashes_com_municipio = gpd.sjoin(gdf_flashes, shapefile_municipios, how='inner', predicate='within')

# Agrupa os resultados por município e estado para contar os flashes
print("Agrupando e contabilizando os flashes...")
contagem_flashes = (flashes_com_municipio.groupby(['name_muni', 'abbrev_state']).size().reset_index(name='Total de Flashes'))

# Renomeia as colunas conforme solicitado
contagem_flashes.rename(columns={'name_muni': 'cidade', 'abbrev_state': 'estado'}, inplace=True)

# Cria uma base de todos os municípios do Brasil para garantir que cidades com zero relâmpagos também apareçam
todos_municipios = pd.DataFrame({'cidade': shapefile_municipios['name_muni'],
                                 'estado': shapefile_municipios['abbrev_state']})

# Mescla todos os municípios com a contagem para preencher com zero onde não houve registro de flashes
planilha_final = pd.merge(todos_municipios, contagem_flashes, on=['cidade', 'estado'], how='left')
planilha_final['Total de Flashes'] = planilha_final['Total de Flashes'].fillna(0).astype(int)

# Ordena do município com mais relâmpagos para o com menos
planilha_final = planilha_final.sort_values(by='Total de Flashes', ascending=False).reset_index(drop=True)

#========================================================================================================================#
#                                       SALVA ARQUIVO CSV FINAL
#========================================================================================================================#
caminho_salvar = f'{dir_estatisticas_estado_e_municipio}/flash_counts_todos_municipios_BR_{anoi}-{mesi}-{diai}.csv'
planilha_final.to_csv(caminho_salvar, index=False)
print(f"Planilha salva em: {caminho_salvar}")

# Exibe as primeiras linhas da planilha final
planilha_final.head(20)

Carregando shapefile de todos os municípios do Brasil...


municipalities_2020_simplified.parquet: 100%|██████████| 20.2M/20.2M [00:00<00:00, 89.5MB/s]


Lendo arquivo de flashes do dia 2026-10-04...
Realizando cruzamento espacial dos flashes com os municípios...
Agrupando e contabilizando os flashes...
Planilha salva em: /content/drive/MyDrive/PYHTON/00_GITHUB/000_CODIGOS_REFERENCIA/03_RELAMPAGOS_SATELITE_REFERENCIA/output/GLM_08/03_estatisticas_estado_e_municipio/flash_counts_todos_municipios_BR_2026-10-04.csv
CPU times: user 10.4 s, sys: 1.14 s, total: 11.5 s
Wall time: 21.9 s


,cidade,estado,Total de Flashes
0,Corumbá,MS,85166
1,Porto Murtinho,MS,59952
2,Aquidauana,MS,39290
3,Apuí,AM,36481
4,Jacareacanga,PA,20193
5,Bonito,MS,17908
6,Ribas do Rio Pardo,MS,16804
7,Novo Aripuanã,AM,16361
8,Miranda,MS,14475
9,Ponta Porã,MS,12839
